## Low-ancilla incrementer

*Dima Fedoriaka, June 2026*

### Summary

Here I present a circuit for incrementing an n-qubit integer register using only O(log n) ancillas.

### Motivation

The practical use case for a low-ancilla incrementer is in constant addition when the constant is small (let's say the bit size of the constant is $n_c$). Then we can use a constant adder circuit with carry on the first $n_c$ qubits. For the rest of the register, we just need to add a carry bit to it, which can be done by applying an incrementer controlled by the carry bit.

### Baseline

As a baseline, I am using the incrementer from [this paper](https://www.worldscientific.com/doi/abs/10.1142/S0217979213501919), which I generalized to a constant adder (https://arxiv.org/pdf/2501.07060) and which is implemented in [ConstAdder.qs](../lib/src/QuantumArithmetic/ConstAdder.qs).

### Implementation idea

Define CTO(x) - "count trailing ones", i.e., the number of least significant bits in x equal to 1 before the first 0 bit.

Then to increment x we need to flip the first CTO(x)+1 bits in x.

So, incrementing is reduced to implementing two operations:
* CountTrailingOnes(x, ans) - computes ans:=CTO(x)
* FlipFirst(target, ctr) - flips first `ctr` bits in `target`.

Both CountTrailingOnes and FlipFirst can be implemented recursively by splitting the input in 2 parts, the first of them having a length equal to a power of 2. Both of them use $O(\log n)$ ancilla, adding one ancilla for each level of recursion.

The incrementer works like this:
* Allocate counter register and carry qubit.
* Compute counter := CTO(x).
* Increment counter using baseline incrementer.
* Compute FlipFirst(x+carry, counter).
* Uncompute carry by applying multi-controlled X, using the fact that carry=1 if and only if counter=Length(x). Note that carry is only needed to handle overflow case when input is 2^n-1. If we can assume it's not going to happen, we don't need carry.
* Uncompute counter by flipping all bits in x, running CTO in reverse and flipping all bits in x again.

The full implementation is in [LAInc.qs](../lib/src/QuantumArithmetic/LAInc.qs) and tests are in [LAInc_test.py](../test/LAInc_test.py).

### Version with carry

To turn the presented incrementer into an incrementer with carry:
* Instead of using ancilla for carry, make it input qubit.
* Do not uncompute the carry qubit.

### Cost

Baseline incrementer uses $n-3$ ancillary qubits.

The presented incrementer uses exactly $2 \lceil \log_2(n+2) \rceil -1$ ancillary qubits, which becomes less than the baseline starting from n=11.

In terms of depth, the proposed circuit uses ~10n CCZ gates while the baseline circuit uses ~1n CCZ gates.

So it's much more expensive in depth, but might be worth it if it can reduce overall space requirement of an algorithm.

The table below compares ancilla count and CCZ count between the baseline and proposed incrementer.

In [1]:
import qdk
import math
ctx = qdk.Context(project_root="../lib/")

headers=["n", "Ancilla (base)", "CCZ (base)", "Ancilla (new)", "CCZ (new)"]

table = []
for n in list(range(1,20))+list(range(20, 110, 10)) + [256]:
    op1 = "QuantumArithmetic.ConstAdder.AddConstant(1L,_)"
    re1 = ctx.logical_counts(f"EstimateUtils.RunUnaryOp({n},{op1})")
    op2 = "QuantumArithmetic.LAInc.IncrementByFlip"
    re2 = ctx.logical_counts(f"EstimateUtils.RunUnaryOp({n},{op2})")    
    anc_new = re2["numQubits"]-n
    assert anc_new == 2*math.ceil(math.log2(n+2)) -1
    table.append([n, re1["numQubits"]-n, re1["cczCount"], anc_new, re2["cczCount"]])


import pandas as pd
pd.DataFrame(table, columns=headers)

,n,Ancilla (base),CCZ (base),Ancilla (new),CCZ (new)
0,1,0,0,3,1
1,2,0,0,3,7
2,3,0,1,5,17
3,4,1,2,5,22
4,5,2,3,5,26
5,6,3,4,5,33
6,7,4,5,7,47
7,8,5,6,7,54
8,9,6,7,7,58
9,10,7,8,7,65
